# 02: Data Profiling

This notebook inspects the structure and content of the merged UNSW-NB15 dataset produced by notebook 01. No modifications are made to the data — the purpose is purely to document what is present before any cleaning takes place.

The profiling covers dataset shape, feature names, data types, summary statistics, missing values, infinite values, and class distributions for both the binary label and the attack category target. Understanding these properties is essential for making informed decisions about which cleaning and transformation steps are needed in notebook 03.

## 2.1: Import Libraries

This section imports all libraries required by the notebook and configures global display settings. NumPy is needed for detecting infinite values, and pandas is used for all data loading and inspection operations. The display option prevents pandas from truncating long column values in any outputs.

In [ ]:
import numpy as np
import pandas as pd

# Prevent truncation of long column values in displayed DataFrames.
pd.set_option('display.max_colwidth', None)

## 2.2: Load Dataset

This section loads the merged dataset produced by notebook 01 from `data/processed/UNSW-NB15.csv`. The `'-'` character is treated as a missing value to be consistent with how the raw files were loaded in the merging step. The first five rows are displayed to confirm that the file has loaded correctly and that column names are present.

In [ ]:
# Load the merged dataset. '-' values are treated as missing to remain consistent with the raw data loading.
df = pd.read_csv('../data/processed/UNSW-NB15.csv', na_values=['-'], low_memory=False)

In [ ]:
# Display the first five rows to confirm the dataset loaded correctly with column names.
df.head()

## 2.3: Shape

This section reports the total number of records and features in the dataset. The expected output is approximately 2.5 million records across 49 columns. Any significant deviation from this would indicate a problem with the merging step in notebook 01.

In [ ]:
# Report the total number of records and features in the dataset.
number_of_records = df.shape[0]
number_of_features = df.shape[1]
print(f"Number of records: {number_of_records:,}")
print(f"Number of features: {number_of_features}")

## 2.4: Feature Names

This section lists all column names in the dataset. Verifying the feature names confirms that the column headers were applied correctly during the merging step and that no names were duplicated, dropped, or misspelled. This list also serves as a reference when selecting or dropping features in later notebooks.

In [ ]:
# Display the full list of column names to verify headers were applied correctly during merging.
df.columns

## 2.5: Data Types and Non-Null Counts

This section calls `df.info()` to display the inferred data type and non-null count for each column. This reveals which columns contain mixed or unexpected types, and which have a significant number of missing values. Columns with dtype `object` that should be numeric are of particular interest as they may contain the `'-'` placeholder that was not fully caught during loading.

In [ ]:
# Display data types and non-null counts for each column.
df.info()

## 2.6: Summary Statistics

This section calls `df.describe()` to display the count, mean, standard deviation, minimum, quartiles, and maximum for each numeric column. These statistics give a first impression of scale differences between features, potential outliers (extreme minimum or maximum values), and features that may benefit from log transformation or standardisation during feature engineering.

In [ ]:
# Display summary statistics for all numeric columns.
df.describe()

## 2.7: Missing Values

This section counts the number of missing values in each column and shows only those columns where missing values are present. Missing values can arise from the `'-'` placeholder in the raw data, or from records where a field was simply absent. These columns will need to be handled explicitly in the cleaning step of notebook 03.

In [ ]:
# Count missing values per column and display only those with at least one missing value.
missing_values = df.isna().sum()
missing_values[missing_values > 0]

## 2.8: Infinite Values

This section counts the total number of infinite values across all numeric columns. Infinite values can cause issues with certain machine learning algorithms and with standardisation, since infinite values have no valid mean or standard deviation. If any are found, they will need to be addressed during cleaning.

In [ ]:
# Count the total number of infinite values across all numeric columns.
infinite_count = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
print(f"Number of infinite values: {infinite_count}")

## 2.9: Class Distribution

This section reports the distribution of records across the binary label (normal vs attack) and the multi-class attack category target. An understanding of class imbalance is essential before modelling — a heavily skewed distribution will require oversampling, undersampling, or class-weighted training to avoid a model that simply predicts the majority class. The attack category distribution also reveals which attack types are rare and may be harder to classify reliably.

In [ ]:
# Display the count of records for each binary label value (0 = normal, 1 = attack).
df["Label"].value_counts()

In [ ]:
# Display the count of records for each attack category.
df["attack_cat"].value_counts()